# 🎯 AI Recruitment: CV vs JD Semantic Matcher (Advanced R&D)

Notebook này phân tích chuyên sâu độ phù hợp giữa CV (File) và JD (Text) dựa trên 2 phương pháp:
1. **Traditional Matching**: Vector Similarity + Rule-based (Fast & Efficient)
2. **AI Deep Reasoning**: Google Gemini LLM (Smart & Context-aware)

In [ ]:
import fitz
import pytesseract
import os, io, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageEnhance, ImageOps
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# --- CONFIG ---
TESS_PATH = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
if os.path.exists(TESS_PATH):
    pytesseract.pytesseract.tesseract_cmd = TESS_PATH
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
SKILL_LIBRARY = ["python", "java", "javascript", "react", "node", "docker", "kubernetes", "aws", "sql", "nosql", "mongodb", "django", "fastapi"]

print("Loading AI Models...")
model = SentenceTransformer(MODEL_NAME)
print("System Ready.")

## 1. Extraction Helpers

In [ ]:
def extract_text(file_path):
    ext = file_path.split('.')[-1].lower()
    text = ""
    if ext == 'pdf':
        doc = fitz.open(file_path)
        for page in doc:
            native = page.get_text().strip()
            if native: text += native + "\n"
            else:
                pix = page.get_pixmap(matrix=fitz.Matrix(400/72, 400/72))
                text += pytesseract.image_to_string(Image.open(io.BytesIO(pix.tobytes("png"))), lang="vie+eng")
        doc.close()
    return text.strip()

def get_skills(text):
    t = text.lower()
    return [s for s in SKILL_LIBRARY if s in t]

def get_exp(text):
    matches = re.findall(r'(\d+)\s*(năm|year)', text.lower())
    years = [int(m[0]) for m in matches]
    return max(years) if years else 0

## 2. Traditional Analysis & Scoring

In [ ]:
def analyze_match(cv_text, jd_text):
    # 1. Semantic (Vector)
    emb = model.encode([cv_text, jd_text])
    sem_score = cosine_similarity([emb[0]], [emb[1]])[0][0]
    
    # 2. Skills
    jd_s = get_skills(jd_text)
    cv_s = get_skills(cv_text)
    matched = [s for s in jd_s if s in cv_s]
    skill_score = len(matched)/len(jd_s) if jd_s else 1.0
    
    # 3. Exp
    cv_e = get_exp(cv_text)
    jd_e = get_exp(jd_text)
    exp_score = min(cv_e/jd_e, 1.0) if jd_e > 0 else 1.0
    
    final = (sem_score * 0.5 + skill_score * 0.3 + exp_score * 0.2) * 100
    return {
        "total": round(final, 2),
        "semantic": round(sem_score * 100, 2),
        "skills": round(skill_score * 100, 2),
        "experience": round(exp_score * 100, 2),
        "matched_list": matched
    }

## 3. Visualization Helpers

In [ ]:
def plot_radar(data, name):
    labels = ['Semantic', 'Skills', 'Experience']
    stats = [data['semantic'], data['skills'], data['experience']]
    
    angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist()
    stats += stats[:1]
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(4, 4), subplot_kw=dict(polar=True))
    ax.fill(angles, stats, color='blue', alpha=0.25)
    ax.plot(angles, stats, color='blue', linewidth=2)
    ax.set_yticklabels([])
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels)
    plt.title(f"Radar Chart: {name}")
    plt.show()

## 4. Run Traditional Analysis

In [ ]:
JD_INPUT = """ Tuyển Python Developer, 3 năm kinh nghiệm, biết Docker, AWS và SQL. """
CV_FOLDER = r"D:\Work\DATN\AI\CV"

all_res = []
if os.path.exists(CV_FOLDER):
    for f in os.listdir(CV_FOLDER):
        if f.endswith('.pdf'):
            try:
                txt = extract_text(os.path.join(CV_FOLDER, f))
                res = analyze_match(txt, JD_INPUT)
                res['name'] = f
                all_res.append(res)
                # plot_radar(res, f) # Uncomment to see radar charts
            except Exception as e:
                print(f"Error processing {f}: {e}")

    if all_res:
        df = pd.DataFrame(all_res).sort_values('total', ascending=False)
        plt.figure(figsize=(10, 4))
        sns.barplot(x='total', y='name', data=df, palette='magma')
        plt.title("Overall Match Scores (Traditional)")
        plt.show()
        display(df[['name', 'total', 'semantic', 'skills', 'experience']])
else:
    print(f"CV Folder not found at {CV_FOLDER}")

## 5. 🤖 Advanced AI Analysis (Google Gemini)

Sử dụng LLM để hiểu ngữ cảnh sâu và đưa ra đánh giá như một HR Manager chuyên nghiệp.

In [ ]:
import google.generativeai as genai
import json

# --- Gemini Config ---
# Lấy API Key tại: https://aistudio.google.com/
GOOGLE_API_KEY = "YOUR_API_KEY_HERE"
genai.configure(api_key=GOOGLE_API_KEY)
llm_model = genai.GenerativeModel('gemini-1.5-flash')

def analyze_with_llm(cv_text, jd_text):
    prompt = f"""
    Bạn là một Senior Technical Recruiter chuyên nghiệp.
    Hãy đánh giá mức độ phù hợp của ứng viên dựa trên CV và JD dưới đây.
    
    [JD]: {jd_text}
    [CV]: {cv_text}
    
    Trả về kết quả duy nhất định dạng JSON (không có text giải thích bên ngoài):
    {{
        "match_score": (0-100),
        "summary": "Tóm tắt ngắn gọn 2 câu về ứng viên",
        "pros": ["Điểm mạnh 1", "Điểm mạnh 2"],
        "cons": ["Điểm yếu 1", "Điểm yếu 2"],
        "verdict": "Shortlist/Consider/Reject",
        "questions": ["Câu hỏi phỏng vấn 1", "Câu hỏi phỏng vấn 2"]
    }}
    """
    
    try:
        response = llm_model.generate_content(prompt)
        clean_json = response.text.replace('```json', '').replace('```', '').strip()
        return json.loads(clean_json)
    except Exception as e:
        return {"error": str(e)}

print("LLM Engine Ready.")

In [ ]:
if 'all_res' in locals() and all_res:
    # Phân tích ứng viên có điểm cao nhất theo cách cũ
    top_candidate = all_res[0]
    sample_cv_path = os.path.join(CV_FOLDER, top_candidate['name'])
    sample_cv_text = extract_text(sample_cv_path)
    
    print(f"--- Phân tích chuyên sâu ứng viên: {top_candidate['name']} ---")
    llm_result = analyze_with_llm(sample_cv_text, JD_INPUT)
    
    print(f"\n[SO SÁNH KẾT QUẢ]")
    print(f"📍 Điểm Cách Cũ (Toán học): {top_candidate['total']}% ")
    print(f"🚀 Điểm LLM (Tư duy AI): {llm_result.get('match_score')}% ")
    
    print(f"\n[ĐÁNH GIÁ CHI TIẾT CỦA AI]")
    print(f"📝 Tóm tắt: {llm_result.get('summary')}")
    print(f"✅ Ưu điểm: {', '.join(llm_result.get('pros', []))}")
    print(f"❌ Nhược điểm: {', '.join(llm_result.get('cons', []))}")
    print(f"⚖️ Kết luận: {llm_result.get('verdict')}")
    print(f"💡 Câu hỏi phỏng vấn gợi ý: \n- " + '\n- '.join(llm_result.get('questions', [])))
else:
    print("Vui lòng chạy Section 4 để có danh sách ứng viên trước.")

## 6. 🔍 Semantic Search Demo

Giả lập tính năng tìm kiếm ứng viên trong kho dữ liệu (Database) sử dụng Vector Embeddings.

In [ ]:
# Giả lập Database ứng viên
candidates = [
    {"name": "Nguyễn Văn A", "summary": "Chuyên gia Python, AI, Machine Learning với 5 năm kinh nghiệm."},
    {"name": "Trần Thị B", "summary": "Frontend Developer chuyên ReactJS, VueJS và thiết kế UI/UX."},
    {"name": "Lê Văn C", "summary": "Data Engineer chuyên về Big Data, Hadoop, Spark và SQL."},
    {"name": "Phạm Văn D", "summary": "Backend Developer chuyên Java Spring Boot và Microservices."}
]

def semantic_search(query, top_k=2):
    print(f"--- Đang tìm kiếm: \"{query}\" ---\n")
    
    # 1. Chuyển Query và Database sang Vector
    candidate_texts = [c["summary"] for c in candidates]
    candidate_embeddings = model.encode(candidate_texts)
    query_embedding = model.encode([query])
    
    # 2. Tính toán độ tương đồng (Cosine Similarity)
    similarities = cosine_similarity(query_embedding, candidate_embeddings)[0]
    
    # 3. Sắp xếp kết quả
    results = []
    for i, score in enumerate(similarities):
        results.append({"candidate": candidates[i], "score": round(float(score) * 100, 2)})
    
    results = sorted(results, key=lambda x: x["score"], reverse=True)
    
    for i, res in enumerate(results[:top_k]):
        print(f"{i+1}. {res["candidate"]["name"]} - Độ phù hợp: {res["score"]}%")
        print(f"   Mô tả: {res["candidate"]["summary"]}\n")

# --- TEST DEMO ---
semantic_search("Tôi cần một người biết làm trí tuệ nhân tạo")
print("-" * 50)
semantic_search("Tìm lập trình viên làm giao diện người dùng")